In [ ]:
import os
import sys
import json

from pathlib import Path
import pandas as pd

path_proj = Path.cwd().parent
path_data = os.path.join(path_proj, "data")

sys.path.append(os.path.join(path_proj, "utils"))

from generate_llm import generate_gpt, get_embeddings





%load_ext autoreload
# %autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
prompt = """
<persona>
You are an economist specializing in monetary policy and central bank communication.
</persona>

<context>
Federal Reserve publication.

Title:
{title}

Publication date:
{publication_date}

Text:
{text}
</context>

<task>
Extract monetary policy signals from the publication and classify each signal according to the monetary policy stance it implies.

Do not extract points that are merely administrative, procedural, biographical, or unrelated to monetary policy.

Classify each key point using one of the following labels:
- dovish: Clearly supports monetary easing or maintaining an accommodative stance. Use when the point indicates that inflation is at target, below target, or moving sustainably toward target; inflation risks are diminishing; economic activity is weakening; or labor market conditions are softening in a way that reduces the need for restrictive policy.
- mostly_dovish: Leans toward monetary easing, a less restrictive stance, or delaying further tightening, but the signal is not definitive. Use when the point reports progress on inflation, slower growth, or softer labor market conditions, while also mentioning uncertainty, remaining inflation risks, mixed data, or the need for further evidence before easing.
- neutral: Policy-relevant but does not clearly imply either easing or tightening. Use when the point is balanced, descriptive, conditional, data-dependent, or uncertain, and when the evidence does not indicate a dominant direction for policy. Do not classify a statement as neutral merely because it is factual.
- mostly_hawkish: Leans toward monetary tightening, maintaining a restrictive stance, or delaying easing, but the signal is not definitive. Use when the point reports elevated inflation, upside inflation risks, resilient demand, or strong labor market conditions, while also acknowledging some moderation, uncertainty, mixed data, or signs of slowing.
- hawkish: Clearly supports monetary tightening or maintaining a restrictive stance. Use when the point indicates that inflation is above target, persistent, broad-based, or rising; inflation expectations are at risk; economic activity remains strong; or labor market conditions remain tight in a way that justifies restrictive policy.


Guidelines:
- Each point should be concise and atomic, expressing one main idea.
- Avoid duplicate or highly overlapping points.
- Evidence must be a short excerpt copied from the text.
- If a category has no relevant points, return an empty list.

Return a JSON object with the following structure:

{
  "dovish": [
    {
      "point": "concise key point",
      "justification": "brief explanation of why this point is classified as dovish",
      "evidence": "short supporting excerpt from the text",
    }
  ],
  "mostly_dovish": [
    {
      "point": "concise key point",
      "justification": "brief explanation of why this point is classified as mostly_dovish",
      "evidence": "short supporting excerpt from the text",
    }
  ],
  "neutral": [
    {
      "point": "concise key point",
      "justification": "brief explanation of why this point is classified as neutral",
      "evidence": "short supporting excerpt from the text",
    }
  ],
  "mostly_hawkish": [
    {
      "point": "concise key point",
      "justification": "brief explanation of why this point is classified as mostly_hawkish",
      "evidence": "short supporting excerpt from the text",
    }
  ],
  "hawkish": [
    {
      "point": "concise key point",
      "justification": "brief explanation of why this point is classified as hawkish",
      "evidence": "short supporting excerpt from the text",
    }
  ]
}
</task>
"""

In [4]:
score_map = {
    "dovish": -1,
    "mostly_dovish": -0.5,
    "neutral": 0,
    "mostly_hawkish": 0.5,
    "hawkish": 1
}

In [ ]:
links_text_path = os.path.join(path_data, "links_text.txt")
result_analysis_path = os.path.join(path_data, "result_analysis.csv")

with open(links_text_path, "r", encoding="utf-8") as f:
    links_text = json.load(f)


try:
    df_result_all = pd.read_csv(result_analysis_path)
except FileNotFoundError:
    df_result_all = pd.DataFrame(columns=["url",
                                          "title",
                                          "publication_date",
                                          "classification",
                                          "score",
                                          "key_point",
                                          "justification",
                                          "evidence",
                                          "embedding_key_point"])

number_items_to_process = 1
counter = 0

for item in links_text:
    counter += 1
    if counter > number_items_to_process:
        break

    url = item["url"]
    title = item["title"]
    text = item["text"]
    publication_date = item["publication_date"]

    if url in df_result_all["url"].values:
        print(f"Skipping already processed URL: {url}")
        continue

    print(f"Processing: {title}")
    print(f"Publication Date: {publication_date}")

    prompt_filled = (
        prompt.replace("{title}", title)
              .replace("{text}", text)
              .replace("{publication_date}", publication_date)
    )

    # print(prompt_filled)

    llm_parameters = {
        "model": "gpt-5.4",
        "json_mode": True,
        "temperature": 0,
        "top_p": 0
    }

    response = generate_gpt(prompt_filled, llm_parameters)

    print(f"Response: {response}")

    response_json = json.loads(response)
    rows = []
    for classification, data in response_json.items():
        print(f"Classification: {classification}")
        print(f"Data: {data}")

        if not data:
            continue

        for item in data:
            key_point = item["point"]
            evidence = item["evidence"]

            print(f"Key Points: {key_point}")
            print(f"Evidence: {evidence}")

            embedding = get_embeddings(key_point)
            rows.append({
                "url": url,
                "title": title,
                "publication_date": publication_date,
                "classification": classification,
                "score": score_map[classification],
                "key_point": key_point,
                "justification": item["justification"],
                "evidence": evidence,
                "embedding_key_point": embedding
            })

    df_result = pd.DataFrame(rows)


df_result_all = pd.concat([df_result_all, df_result], ignore_index=True)
df_result_all.to_csv(result_analysis_path, index=False)

Processing: Federal Reserve issues FOMC statement
Publication Date: 2026-01-28
Response: {
  "dovish": [
    {
      "point": "Two FOMC members preferred an immediate 25 basis point rate cut.",
      "justification": "An explicit preference for lowering the policy rate is a clear easing signal.",
      "evidence": "preferred to lower the target range for the federal funds rate by 1/4 percentage point at this meeting"
    }
  ],
  "mostly_dovish": [
    {
      "point": "Labor market conditions appear softer, with low job gains and unemployment stabilizing.",
      "justification": "Weaker hiring and signs that unemployment is no longer falling lean toward easing, though not decisively because the statement does not describe severe deterioration.",
      "evidence": "Job gains have remained low, and the unemployment rate has shown some signs of stabilization"
    }
  ],
  "neutral": [
    {
      "point": "The Committee left the federal funds rate unchanged at 3.5% to 3.75%.",
      "ju

: 